# 0.Library

In [80]:
import pandas as pd
from pathlib import Path

import importlib
import sys
sys.path.append('../') 
from features import data_utils as du
from features import data_pipeline as dp

# Reload
importlib.reload(du)
importlib.reload(dp)

# Constants and paths
parent_folder = Path("../..") # go 2 folder up: "../.."
data_folder = parent_folder / "data"
input_path_of_json = data_folder / "patients_data_log.json"
df_data_log = pd.read_json(input_path_of_json)

patient_ids = df_data_log["PatientID"].to_list()
patients_data_folder = data_folder / "PatientsData"

# feature vectors

In [81]:
# feature vectors
hover_dict = {
    "total_hover_count": None,
    "total_hover_duration": None,
    "mean_hover_duration": None,
    "max_hover_duration": None,
    "median_hover_duration": None,
    "std_hover_duration": None,
    "hover_intensity": None
}

press_dict = {
    "total_press_count": None,
    "total_press_duration": None,
    "mean_press_duration": None,
    "max_press_duration": None,
    "median_press_duration": None,
    "std_press_duration": None,
    "press_intensity": None # Interaction intensity
}

reading_time_dict = {
    "total_reading_time_duration": None,
    "mean_reading_time_duration": None,
    "max_reading_time_duration": None,
    "median_reading_time_duration": None,
    "std_reading_time_duration": None,
    "reading_time_intensity" : None
}

merged_ratio_dict = {
    "hover_vs_active_interaction_ratio": None,
    "hover_vs_reading_time_ratio": None
}

# get clean path of one patient (all csv files)

In [82]:
# get clean path of one patient (all csv files)
patient_id = patient_ids[0]
patient_folder_path = du.find_patient_folder(patients_data_folder=patients_data_folder,
                                               patient_id=patient_id)
                                               
path_to_check = patient_folder_path / "clean_data"
clean_csv_files = du.find_csv_file(patient_folder_path / "clean_data")
clean_csv_files

['cleaned_20260502_113117_01_Tutorial_events.csv',
 'cleaned_20260502_113117_01_Tutorial_trajectory.csv',
 'cleaned_20260502_113117_02_ObjectRecognition_events.csv',
 'cleaned_20260502_113117_02_ObjectRecognition_trajectory.csv',
 'cleaned_20260502_113117_03_Visuospatial_events.csv',
 'cleaned_20260502_113117_03_Visuospatial_trajectory.csv',
 'cleaned_20260502_113117_04_Memory_events.csv',
 'cleaned_20260502_113117_04_Memory_trajectory.csv']

In [83]:
# start with the first csv file
clean_df = pd.read_csv(path_to_check / clean_csv_files[0])

In [84]:
clean_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.0,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.0,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


In [85]:
clean_df.columns

Index(['Timestamp', 'PhaseTime_s', 'Section', 'EventType', 'ObjectName',
       'Hand', 'IsCorrect', 'Duration_s', 'Distance_m', 'Details',
       'Activity_Log'],
      dtype='object')

# work on one section

In [86]:
# work on one section
section_names = clean_df['Section'].unique()
section_names

array(['Default', 'ButtonSelection', 'GrabPractice', 'Completion'],
      dtype=object)

In [87]:
section_df = du.extract_section(clean_df, section_names[0])
section_df.head()

,Timestamp,PhaseTime_s,Section,EventType,ObjectName,Hand,IsCorrect,Duration_s,Distance_m,Details,Activity_Log
0,2026-05-02 11:31:17.628,0.000,Default,PHASE_START,NaN,NaN,NaN,0.0,0.0,Phase=01_Tutorial,"['Section=Default', 'SessionID=20260502_113117..."
1,2026-05-02 11:31:17.630,0.000,Default,PANEL_SHOWN,WelcomePanel,NaN,NaN,0.0,0.0,Section=Default,['Panel=WelcomePanel']
2,2026-05-02 11:31:22.971,0.000,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
3,2026-05-02 11:31:23.918,0.952,Default,BUTTON_HOVER_END,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...
4,2026-05-02 11:31:26.072,3.106,Default,BUTTON_HOVER_START,NaN,NaN,NaN,0.0,0.0,Section=Default,['Button=StartTutorialButton[TutorialWelcome]'...


# extract function

In [88]:
def extract_section_duration(row_section_end):
    import re
    match = re.search(r"Duration=([0-9]*\.?[0-9]+)", row_section_end)
    return float(match.group(1)) if match else None

In [89]:
def extract_hover_count(s):
    import re
    match = re.search(r"HoverCount=(\d+)", s)
    return int(match.group(1)) if match else None

In [90]:
def extract_hover_duration(s):
    import re
    match = re.search(r"HoverDuration=([0-9]*\.?[0-9]+)", s)
    return float(match.group(1)) if match else None

In [91]:
def extract_press_count(s):
    pass

In [92]:
def extract_press_duration(s):
    import re
    match = re.search(r"PressDuration=([0-9]*\.?[0-9]+)", s)
    return float(match.group(1)) if match else None

In [93]:
def extract_reading_time_duration(s):
    import re
    match = re.search(r"ReadingTime=([0-9]*\.?[0-9]+)", s)
    return float(match.group(1)) if match else None

In [94]:
def extract_section_duration(s):
    import re
    match = re.search(r"Duration=([0-9]*\.?[0-9]+)", s)
    return float(match.group(1)) if match else None

In [95]:
def extract_metric_from_section(s, function_to_extract_metric):
    section_metric_values = []
    for value in s:
        print(function_to_extract_metric(value))
        section_metric_values.append(function_to_extract_metric(value))

    section_metric_values_pandas = pd.Series(section_metric_values)
    return section_metric_values_pandas


# filter function

In [96]:
def filter_by_string_contains(df, column_name, substring):
    return df[df[column_name].str.contains(substring, na=False)]

# calculate_metric

In [97]:
def calculate_metric_stats(metric_series, is_counting=True):

    # calculate total sum for both counting and duration metrics
    total = round(metric_series.sum(),2)
    print("Sum:", total)

    # For counting metrics, we only care about the total sum.
    if is_counting:
        total = metric_series.sum()
        return total, None, None, None, None
    
    # For duration metrics, we calculate all statistics.
    mean = round(metric_series.mean(), 2)
    maximum = round(metric_series.max(), 2)
    median = round(metric_series.median(), 2)
    std = round(metric_series.std(), 2)

    print("Mean:", mean)
    print("Max:", maximum)
    print("Median:", median)
    print("Std:", std)

    return total, mean, maximum, median, std

# start the calculation

## cal for hover

In [98]:
# hover count
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'HoverCount')
hover_count_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_hover_count)

# hover_count
hover_dict["total_hover_count"], _ , _ , _ , _ = calculate_metric_stats(hover_count_series, is_counting=True)


1
2
3
4
5
6
7
Sum: 28


In [99]:
# hover duration
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'HoverDuration')
hover_duration_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_hover_duration)

# calculation for hover duration
hover_dict["total_hover_duration"], hover_dict["mean_hover_duration"], hover_dict["max_hover_duration"], hover_dict["median_hover_duration"], hover_dict["std_hover_duration"] = calculate_metric_stats(hover_duration_series, is_counting=False)

0.95
0.11
0.32
0.54
2.29
0.49
0.82
Sum: 5.52
Mean: 0.79
Max: 2.29
Median: 0.54
Std: 0.72


In [100]:
hover_dict

{'total_hover_count': np.int64(28),
 'total_hover_duration': np.float64(5.52),
 'mean_hover_duration': np.float64(0.79),
 'max_hover_duration': np.float64(2.29),
 'median_hover_duration': np.float64(0.54),
 'std_hover_duration': np.float64(0.72),
 'hover_intensity': None}

## cal for press

In [101]:
# calculation for press count (assumed that for each button pressed, there is a button released too
filtered_df_button_pressed = filter_by_string_contains(section_df, 'EventType', 'BUTTON_PRESSED')
total_press_count = len(filtered_df_button_pressed)  # Assuming each press has a corresponding release
press_dict["total_press_count"] = total_press_count

In [102]:
# press duration
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'PressDuration')
press_duration_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_press_duration)

press_dict["total_press_duration"], press_dict["mean_press_duration"], press_dict["max_press_duration"], press_dict["median_press_duration"], press_dict["std_press_duration"] = calculate_metric_stats(press_duration_series, is_counting=False)

0.222
Sum: 0.22
Mean: 0.22
Max: 0.22
Median: 0.22
Std: nan


In [103]:
press_dict

{'total_press_count': 1,
 'total_press_duration': np.float64(0.22),
 'mean_press_duration': np.float64(0.22),
 'max_press_duration': np.float64(0.22),
 'median_press_duration': np.float64(0.22),
 'std_press_duration': np.float64(nan),
 'press_intensity': None}

## cal for reading time

In [104]:
# reading time duration
filtered_df = filter_by_string_contains(section_df, 'Activity_Log', 'ReadingTime')
reading_time_series = extract_metric_from_section(filtered_df['Activity_Log'], extract_reading_time_duration)

# calculation for reading time duration
reading_time_dict["total_reading_time_duration"], reading_time_dict["mean_reading_time_duration"], reading_time_dict["max_reading_time_duration"], reading_time_dict["median_reading_time_duration"], reading_time_dict["std_reading_time_duration"] = calculate_metric_stats(reading_time_series, is_counting=False)

17.23
Sum: 17.23
Mean: 17.23
Max: 17.23
Median: 17.23
Std: nan


In [105]:
# extract section duration
row_section_end = section_df[section_df['EventType']=='SECTION_END']['Activity_Log'].values[0]

section_total_time = extract_section_duration(row_section_end)

# Ratio functions

In [116]:
def ratio_calculation( value1, value2):
    if value2 == 0:
        return None  # or you could return float('inf') to indicate an infinite ratio
    return round(value1 / value2, 2)

In [117]:
# hover intensity
hover_dict["hover_intensity"] = ratio_calculation(hover_dict["total_hover_duration"], section_total_time)

In [118]:
press_dict["press_intensity"] = ratio_calculation(press_dict["total_press_duration"], section_total_time)

In [119]:
reading_time_dict["reading_time_intensity"] = ratio_calculation(reading_time_dict["total_reading_time_duration"], section_total_time)

In [120]:
merged_ratio_dict["hover_vs_active_interaction_ratio"] = ratio_calculation(hover_dict["total_hover_duration"], press_dict["total_press_duration"])
merged_ratio_dict["hover_vs_reading_time_ratio"] = ratio_calculation(hover_dict["total_hover_duration"], reading_time_dict["total_reading_time_duration"])

# final vectors

In [121]:
hover_dict

{'total_hover_count': np.int64(28),
 'total_hover_duration': np.float64(5.52),
 'mean_hover_duration': np.float64(0.79),
 'max_hover_duration': np.float64(2.29),
 'median_hover_duration': np.float64(0.54),
 'std_hover_duration': np.float64(0.72),
 'hover_intensity': np.float64(0.32)}

In [122]:
press_dict

{'total_press_count': 1,
 'total_press_duration': np.float64(0.22),
 'mean_press_duration': np.float64(0.22),
 'max_press_duration': np.float64(0.22),
 'median_press_duration': np.float64(0.22),
 'std_press_duration': np.float64(nan),
 'press_intensity': np.float64(0.01)}

In [123]:
reading_time_dict

{'total_reading_time_duration': np.float64(17.23),
 'mean_reading_time_duration': np.float64(17.23),
 'max_reading_time_duration': np.float64(17.23),
 'median_reading_time_duration': np.float64(17.23),
 'std_reading_time_duration': np.float64(nan),
 'reading_time_intensity': np.float64(1.0)}

In [124]:
merged_ratio_dict

{'hover_vs_active_interaction_ratio': np.float64(25.09),
 'hover_vs_reading_time_ratio': np.float64(0.32)}